# Fluxo de Autorização — analise_siplan_rps + prog_enviada

Acionado pelo **Power Automate por botão no Power BI**. 

## O que faz

1. Busca dados das atividades em `lake_gold_fatos.dbo.base`.
2. Registra/atualiza `wh_siplan_rps.dbo.analise_siplan_rps`:

| Condição | Ação |
|----------|------|
| `atividade_id` não existe | INSERT completo, Status via `get_status_inicial(autonomia)` |
| existe com Status em `STATUSES_ATUALIZAVEIS` | UPDATE (exceto `custos_foto`) |
| existe com outro Status | ignorado, registrado em log |

4. Grava linha em `lake_relatorios_gerados.dbo.prog_enviada`.

## Retorno para o Power Automate

```json
{"gravado": true, "resultado_approval": "Completo", "inseridos": 2, "atualizados": 1, "ignorados": 0}
```
ou, se resultado não gravável:
```json
{"gravado": false, "resultado_approval": "Não aprovado"}
```

> **`nb_EntregaProgs` permanece inalterado** quanto ao arquivo .txt e retorno JSON.
> `registrar_relatorio()` foi removido de lá — o log de entrega é gravado aqui,
> após confirmação do Approval.

In [ ]:
atividade_ids_str   = '63000016479883,63000018896693,63000018461299'
solicitante         = ''   # email de quem solicitou (injetado pelo PA)

ANALISE_TABLE = 'wh_siplan_rps.dbo.analise_siplan_rps'

SQL_ENDPOINT  = (
    'beu5bmmdbuwedpv62ucm524jzi-dmrv7k3fbwbevh5d4sidg3urfq'
    '.datawarehouse.fabric.microsoft.com'
)

# ── Resultado do Approval (injetado pelo PA) ─────────────────────────────────
resultado_approval   = 'Completo'   # Completo | Parcial | Não aprovado | Sem retorno
comentario_approval  = ''           # comentário livre do aprovador

# ── Dados do relatório gerado por nb_EntregaProgs (injetados pelo PA) ────────
relatorio_id    = ''
relatorio_url   = ''
relatorio_nome  = 'Entrega da Programação para a STS'
gerencias       = ''
qt_total        = 0
data_geracao    = ''   # ISO string — gerado_em retornado por nb_EntregaProgs

# ── Resultados que disparam a gravação ───────────────────────────────────────
# Edite para adicionar/remover resultados válidos sem mexer no restante.
RESULTADOS_GRAVAM = frozenset({
    'Completo',  # aprovação integral
    'Parcial',   # aprovação com ressalvas
})

# ── Mapeamento autonomia → Status no INSERT ──────────────────────────────────
# Edite este dicionário para adicionar/remover regras sem mexer no código.
STATUS_POR_AUTONOMIA = {
    'UO': 'AutonomiaUO',   # autonomia da UO → não vai para STS
}
STATUS_PADRAO = 'Enviado'  # Status padrão quando autonomia não está mapeada

# ── Status que permitem UPDATE quando a linha já existe ──────────────────────
# Linhas com outros Status são ignoradas (log apenas).
STATUSES_ATUALIZAVEIS = frozenset({
    'Enviado',    # entrega inicial registrada
    'Reenviado',  # correção de entrega anterior
    'Em Revisão', # aguardando parecer
})

In [ ]:
# ── Imports e detecção de ambiente ───────────────────────────────────────────
import json
import struct
import warnings
from datetime import datetime

import pandas as pd

warnings.filterwarnings('ignore')

try:
    spark
    FABRIC_ENV = True
    print('Ambiente: Microsoft Fabric')
except NameError:
    FABRIC_ENV = False
    print('Ambiente: local')

try:
    from notebookutils import mssparkutils as _ms
    mssparkutils = _ms
    HAS_MSSPARKUTILS = True
except ImportError:
    HAS_MSSPARKUTILS = False

if not FABRIC_ENV:
    try:
        import pyodbc
    except ImportError:
        print('[AVISO] pyodbc não disponível — instale via pip install pyodbc')
        raise

print(f'Imports OK — {datetime.now():%d/%m/%Y %H:%M}')

In [ ]:
# ── Conexão com o SQL endpoint (uso local apenas) ────────────────────────────
def _get_db_conn():
    from azure.identity import InteractiveBrowserCredential, DeviceCodeCredential
    try:
        cred = InteractiveBrowserCredential()
    except Exception:
        cred = DeviceCodeCredential()
    token = cred.get_token('https://database.windows.net/.default').token
    tb = token.encode('utf-16-le')
    ts = struct.pack(f'<I{len(tb)}s', len(tb), tb)
    return pyodbc.connect(
        f'DRIVER={{ODBC Driver 17 for SQL Server}};'
        f'SERVER={SQL_ENDPOINT};Encrypt=Yes;',
        attrs_before={1256: ts},
    )


print('Auth helpers definidos.')

In [ ]:
# ── Busca dados das atividades com todos os campos necessários ────────────────
def fetch_atividades(ids: list) -> pd.DataFrame:
    q = chr(39)
    ids_sql = ', '.join(q + str(i) + q for i in ids)

    # Fabric: spark.sql com CAST AS STRING e JOIN cross-lakehouse
    sql_fabric = (
        'SELECT '
        '    b.atividade_id, b.nome, b.custo_total, b.gerencia, '
        '    b.PrimeiraData AS dataPrimeiraSessao, '
        '    b.areaprog AS area, '
        '    b.linguagem, b.mes, b.autonomia, '
        '    b.complemento, b.item_desc, '
        '    b.projeto_nome AS projeto, '
        '    b.precificacao_desc, '
        '    du.unidade, '
        '    MAX(ds.localNome) AS localNome_max '
        'FROM lake_gold_fatos.dbo.base b '
        'LEFT JOIN lake_gold_fatos.dbo.dim_unidade du '
        '       ON CAST(LEFT(CAST(b.atividade_id AS STRING), 2) AS INT) = du.uo '
        'LEFT JOIN lake_gold_siplan.dbo.datas_sessoes ds '
        '       ON b.atividade_id = ds.atividade_id '
        f'WHERE b.atividade_id IN ({ids_sql}) '
        'GROUP BY b.atividade_id, b.nome, b.custo_total, b.gerencia, '
        '         b.PrimeiraData, b.areaprog, b.linguagem, b.mes, b.autonomia, '
        '         b.complemento, b.item_desc, b.projeto_nome, b.precificacao_desc, '
        '         du.unidade'
    )

    # Local: pyodbc com CAST AS VARCHAR e subquery para localNome
    sql_local = (
        'SELECT '
        '    b.atividade_id, b.nome, b.custo_total, b.gerencia, '
        '    b.PrimeiraData AS dataPrimeiraSessao, '
        '    b.areaprog AS area, '
        '    b.linguagem, b.mes, b.autonomia, '
        '    b.complemento, b.item_desc, '
        '    b.projeto_nome AS projeto, '
        '    b.precificacao_desc, '
        '    du.unidade, '
        '    (SELECT MAX(ds.localNome) '
        '       FROM lake_gold_siplan.dbo.datas_sessoes ds '
        '      WHERE ds.atividade_id = b.atividade_id) AS localNome_max '
        'FROM lake_gold_fatos.dbo.base b '
        'LEFT JOIN lake_gold_fatos.dbo.dim_unidade du '
        '       ON CAST(LEFT(CAST(b.atividade_id AS VARCHAR(20)), 2) AS INT) = du.uo '
        f'WHERE b.atividade_id IN ({ids_sql})'
    )

    if FABRIC_ENV:
        df = spark.sql(sql_fabric).toPandas()
    else:
        conn = _get_db_conn()
        df   = pd.read_sql(sql_local, conn)
        conn.close()

    df['atividade_id'] = pd.to_numeric(df['atividade_id'], errors='coerce').astype('Int64')
    df['custo_total']  = pd.to_numeric(df['custo_total'],  errors='coerce').fillna(0.0)
    return df


print('fetch_atividades definida.')

In [ ]:
# ── Helpers: construção de campos e execução de DML ──────────────────────────

def _build_custos(row) -> str:
    """Texto multilinha para custos_foto e custos_editavel."""
    local_prec = (
        str(row.get('localNome_max') or '') + ' - ' + str(row.get('precificacao_desc') or '')
    ).strip(' -')
    partes = [
        str(row.get('dataPrimeiraSessao') or ''),
        str(row.get('complemento') or ''),
        str(row.get('item_desc') or ''),
        str(row.get('projeto') or ''),
        local_prec,
    ]
    return chr(10).join(p for p in partes if p)


def get_status_inicial(autonomia) -> str:
    """Retorna o Status a usar no INSERT com base na autonomia da atividade."""
    return STATUS_POR_AUTONOMIA.get(str(autonomia or '').strip(), STATUS_PADRAO)


def _sql_val(v) -> str:
    """Serializa valor Python para literal T-SQL seguro (sem bind params)."""
    if v is None:
        return 'NULL'
    if isinstance(v, bool):
        return '1' if v else '0'
    if isinstance(v, (int, float)):
        return str(v)
    if isinstance(v, datetime):
        return chr(39) + v.strftime('%Y-%m-%d %H:%M:%S') + chr(39)
    s = str(v).replace(chr(39), chr(39) + chr(39))   # escapa aspas simples
    return chr(39) + s + chr(39)


def _exec_sql(sql: str) -> None:
    """Executa DML. Fabric: spark.sql; local: pyodbc."""
    if FABRIC_ENV:
        spark.sql(sql)
    else:
        conn = _get_db_conn()
        conn.execute(sql)
        conn.commit()
        conn.close()


def _query_existentes(ids_sql: str) -> pd.DataFrame:
    sel = (
        f'SELECT atividade_id, Status '
        f'FROM {ANALISE_TABLE} '
        f'WHERE atividade_id IN ({ids_sql})'
    )
    if FABRIC_ENV:
        return spark.sql(sel).toPandas()
    conn = _get_db_conn()
    df   = pd.read_sql(sel, conn)
    conn.close()
    return df


def registrar_entrega(ids: list, data_ger: datetime) -> None:
    """Grava log de entrega em prog_enviada (com resultado e comentário do Approval)."""
    row = pd.DataFrame([{
        'relatorio_id':        relatorio_id,
        'relatorio_nome':      relatorio_nome,
        'data_geracao':        data_ger,
        'url_arquivo':         relatorio_url,
        'solicitante':         solicitante,
        'atividade_ids':       ','.join(str(i) for i in ids),
        'qt_total':            qt_total,
        'gerencias':           gerencias,
        'resultado_approval':  resultado_approval,
        'comentario_approval': comentario_approval,
    }])
    if FABRIC_ENV:
        (spark.createDataFrame(row)
              .write.mode('append')
              .option('mergeSchema', 'true')
              .saveAsTable('lake_relatorios_gerados.dbo.prog_enviada'))
        print('Linha registrada em lake_relatorios_gerados.dbo.prog_enviada.')
    else:
        print('LOCAL — registro que seria gravado em prog_enviada:')
        print(row.to_string(index=False))


print('Helpers definidos.')

In [ ]:
# ── Lógica principal de INSERT / UPDATE ──────────────────────────────────────
def registrar_analise(df_ativ: pd.DataFrame,
                      data_entrega: datetime, quem: str) -> dict:
    """
    Para cada atividade_id:
      - Não existe → INSERT completo (Status via get_status_inicial)
      - Existe com Status em STATUSES_ATUALIZAVEIS
            → UPDATE campos de entrega + dados da base (exceto custos_foto)
      - Existe com outro Status → ignora, registra em log

    Retorna dict com contagens: inseridos, atualizados, ignorados.
    """
    ids = df_ativ['atividade_id'].dropna().astype('int64').tolist()
    if not ids:
        print('Nenhum atividade_id para processar.')
        return {'inseridos': 0, 'atualizados': 0, 'ignorados': 0}

    ids_sql     = ', '.join(str(i) for i in ids)
    existentes  = _query_existentes(ids_sql)
    status_atual = dict(zip(
        existentes['atividade_id'].astype('int64'),
        existentes['Status'],
    ))

    novos, atualizados, ignorados = [], [], []

    for _, row in df_ativ.iterrows():
        aid    = int(row['atividade_id'])
        custos = _build_custos(row)

        if aid not in status_atual:
            novos.append({
                'atividade_id':       aid,
                'nome':               row.get('nome') or '',
                'Título':             row.get('nome') or '',
                'unidade':            row.get('unidade') or '',
                'gerencia':           row.get('gerencia') or '',
                'area':               row.get('area') or '',
                'linguagem':          row.get('linguagem') or '',
                'mes':                row.get('mes') or '',
                'autonomia':          row.get('autonomia') or '',
                'dataPrimeiraSessao': row.get('dataPrimeiraSessao'),
                'custos_foto':        custos,
                'custos_editavel':    custos,
                'data_entrega':       data_entrega,
                'quem':               quem,
                'Criado por':         quem,
                'Criado':             data_entrega,
                'Status':             get_status_inicial(row.get('autonomia')),
            })

        elif status_atual[aid] in STATUSES_ATUALIZAVEIS:
            atualizados.append({
                'atividade_id':       aid,
                'nome':               row.get('nome') or '',
                'Título':             row.get('nome') or '',
                'unidade':            row.get('unidade') or '',
                'gerencia':           row.get('gerencia') or '',
                'area':               row.get('area') or '',
                'linguagem':          row.get('linguagem') or '',
                'mes':                row.get('mes') or '',
                'autonomia':          row.get('autonomia') or '',
                'dataPrimeiraSessao': row.get('dataPrimeiraSessao'),
                'custos_editavel':    custos,   # custos_foto nunca é atualizado
                'data_entrega':       data_entrega,
                'quem':               quem,
            })

        else:
            ignorados.append((aid, status_atual[aid]))

    # INSERT
    for item in novos:
        cols = ', '.join(f'[{c}]' for c in item)
        vals = ', '.join(_sql_val(v) for v in item.values())
        _exec_sql(f'INSERT INTO {ANALISE_TABLE} ({cols}) VALUES ({vals})')

    # UPDATE
    for item in atualizados:
        aid  = item.pop('atividade_id')
        sets = ', '.join(f'[{c}] = {_sql_val(v)}' for c, v in item.items())
        _exec_sql(
            f'UPDATE {ANALISE_TABLE} '
            f'SET {sets} '
            f'WHERE atividade_id = {aid}'
        )

    counts = {'inseridos': len(novos), 'atualizados': len(atualizados), 'ignorados': len(ignorados)}
    print(f'analise_siplan_rps: {counts}')
    for aid, st in ignorados:
        print(f'  atividade {aid} ignorada (Status={st!r})')
    return counts


print('registrar_analise definida.')

In [ ]:
# ── Execução ──────────────────────────────────────────────────────────────────

# 1. Verifica se o resultado do Approval dispara gravação
if resultado_approval not in RESULTADOS_GRAVAM:
    print(f'Resultado "{resultado_approval}" — nenhum registro gravado.')
    saida = json.dumps(
        {'gravado': False, 'resultado_approval': resultado_approval},
        ensure_ascii=False,
    )
    print(saida)
    if HAS_MSSPARKUTILS:
        mssparkutils.notebook.exit(saida)
    raise SystemExit(0)

# 2. Parseia IDs
raw = atividade_ids_str.strip()
if raw.startswith('{'):
    raw = '[' + raw + ']'
if raw.startswith('['):
    parsed = json.loads(raw)
    if parsed and isinstance(parsed[0], dict):
        ids = [int(x['atividade_id']) for x in parsed]
    else:
        ids = [int(x) for x in parsed]
else:
    ids = [int(x.strip()) for x in raw.split(',') if x.strip()]

print(f'{len(ids)} atividades: {ids}')

# 3. Busca dados e grava analise_siplan_rps
df     = fetch_atividades(ids)
print(f'Dados carregados: {len(df)} linha(s)')
counts = registrar_analise(df, datetime.now(), solicitante)

# 4. Grava prog_enviada (log de entrega com resultado e comentário do Approval)
data_ger = datetime.fromisoformat(data_geracao) if data_geracao else datetime.now()
registrar_entrega(ids, data_ger)

# 5. Retorno para o Power Automate
saida = json.dumps(
    {'gravado': True, 'resultado_approval': resultado_approval, **counts},
    ensure_ascii=False,
)
print(saida)

if HAS_MSSPARKUTILS:
    mssparkutils.notebook.exit(saida)